[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedelshazly27/mizan-kuwait-energy-toolkit/blob/main/notebooks/07-economics.ipynb)

In [1]:
# On Colab: clone the repository and install it (inputs are committed, no credentials needed).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("mizan-kuwait-energy-toolkit"):
    !git clone --depth 1 https://github.com/ahmedelshazly27/mizan-kuwait-energy-toolkit.git
    %pip install -q -e "mizan-kuwait-energy-toolkit[pv,econ,system]" matplotlib
    os.chdir("mizan-kuwait-energy-toolkit")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from kwtk import viz; viz.style()

# 7. Economics

PySAM Single Owner on the lifetime hourly export, with real costs and the Kuwait
WACC. The verified capital costs are NREL's US benchmarks; Gulf costs are lower
(IRENA's global 2024 average and Saudi battery auctions, both unverified). This
notebook shows how much that choice moves the answer.

In [2]:
import yaml, tempfile
from kwtk.plant.evaluate import evaluate
def with_capex(pv_key, label):
    cfg = yaml.safe_load(open('configs/examples/shagaya-scale-pv-tracker.yaml'))
    cfg['economics']['capex'] = {'pv_total_usd_per_kwac': f'assumption:{pv_key}'}
    cfg['id'] = label; cfg['evaluation']['system_context'] = None; cfg['evaluation']['years'] = [2023]
    f = tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False); yaml.safe_dump(cfg, f); f.close()
    return evaluate(f.name)['result']['economics']
rows = {'NREL ATB 2025 (US, verified)': with_capex('capex.pv_atb2025_usd_per_kw', 'atb'),
        'IRENA 2024 global average (unverified)': with_capex('capex.pv_irena2024_global_usd_per_kw', 'irena')}
pd.DataFrame(rows).T[['capex_usd', 'lcoe_real_fils_per_kwh', 'lcoe_real_usd_per_kwh', 'breakeven_ppa_fils_per_kwh']].round(3)

,capex_usd,lcoe_real_fils_per_kwh,lcoe_real_usd_per_kwh,breakeven_ppa_fils_per_kwh
"NREL ATB 2025 (US, verified)",1677191810.2175,16.783156,0.054727,20.334167
IRENA 2024 global average (unverified),759586787.39,9.686234,0.031585,11.735665


For context, reported Gulf PV tariffs are 1.04-2.34 US cents/kWh (not verified). The LCOE uses PySAM; an independent capital-recovery calculation agrees to within 0.3 %.